# Score fusion

In [2]:
import sys
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Union

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path("/nvme2/hungdx/Lightning-hydra").resolve()


import eval_metrics_DF as em  # noqa: E402

IGNORED_PREFIXES = ("merged_protocol", "merged_scores", "pooled_", "summary")
Reducer = Union[str, Callable[[np.ndarray], float]]


def _resolve_repo_path(path_like: Union[str, Path]) -> Path:
    path = Path(path_like)
    if path.is_absolute():
        return path
    return (REPO_ROOT / path).resolve()


def _read_protocol(protocol_path: Path) -> pd.DataFrame:
    if not protocol_path.exists():
        raise FileNotFoundError(f"Protocol file not found: {protocol_path}")
    df = pd.read_csv(
        protocol_path,
        sep=r"\s+",
        header=None,
        names=["path", "subset", "label"],
        dtype={"path": str, "subset": str, "label": str},
    )
    # Only use subset is eval
    df = df[df["subset"] == "eval"]
    return df[["path", "label"]]


def _read_score_file(score_path: Path) -> pd.DataFrame:
    if not score_path.exists():
        raise FileNotFoundError(f"Score file not found: {score_path}")
    df = pd.read_csv(
        score_path,
        sep=r"\s+",
        header=None,
        names=["path", "spoof_score", "bona_score"],
        dtype={"path": str, "spoof_score": np.float32, "bona_score": np.float32},
    )
    return df.drop_duplicates(subset="path")


def _discover_datasets(folder: Path) -> Dict[str, Dict[str, Path]]:
    """Return {dataset_name: {"score": Path, "protocol": Path}} for a folder."""
    dataset_map: Dict[str, Dict[str, Path]] = {}
    pooled_meta = sorted(folder.glob("pooled_merged_protocol*.txt"))

    if pooled_meta:
        meta_path = pooled_meta[0]
        with meta_path.open() as handle:
            for line in handle:
                line = line.strip()
                if (
                    not line
                    or line.startswith("#")
                    or line.startswith("TOTAL")
                    or "|" not in line
                ):
                    continue
                parts = [part.strip() for part in line.split("|")]
                if len(parts) < 4:
                    continue
                dataset_name, _, protocol_str, score_str = parts[:4]
                protocol_path = _resolve_repo_path(protocol_str)
                score_path = _resolve_repo_path(score_str)
                if not score_path.exists():
                    fallback = folder / Path(score_str).name
                    if fallback.exists():
                        score_path = fallback
                dataset_map[dataset_name] = {
                    "protocol": protocol_path,
                    "score": score_path,
                }
        return dataset_map

    # Fallback: infer from filenames (best effort)
    for txt_path in folder.glob("*.txt"):
        if txt_path.name.startswith(IGNORED_PREFIXES):
            continue
        dataset_name = txt_path.stem.split("_", 1)[0]
        protocol_candidate = _resolve_repo_path(
            f"data/wildspoof_challenge_benchmark/{dataset_name}/protocol.txt"
        )
        dataset_map[dataset_name] = {
            "protocol": protocol_candidate,
            "score": txt_path,
        }
    return dataset_map


def _select_columns(prefix: str, columns: Iterable[str]) -> List[str]:
    return [col for col in columns if col.startswith(prefix)]


def _apply_reducer(frame: pd.DataFrame, cols: List[str], reducer: Reducer) -> pd.Series:
    if not cols:
        raise ValueError("No columns provided for aggregation")

    if callable(reducer):
        return frame[cols].apply(lambda row: reducer(row.values.astype(float)), axis=1)

    reducer_key = reducer.lower()
    if reducer_key == "mean":
        return frame[cols].mean(axis=1)
    if reducer_key == "median":
        return frame[cols].median(axis=1)
    if reducer_key == "max":
        return frame[cols].max(axis=1)
    if reducer_key == "min":
        return frame[cols].min(axis=1)
    if reducer_key == "sum":
        return frame[cols].sum(axis=1)

    raise ValueError(f"Unsupported reducer: {reducer}")


def _evaluate_scores(scored_df: pd.DataFrame) -> Dict[str, Any]:
    preds = np.where(scored_df["spoof_score"] < scored_df["bona_score"], "bonafide", "spoof")
    accuracy = float((preds == scored_df["label"].to_numpy()).mean() * 100)

    bona_scores = scored_df.loc[scored_df["label"] == "bonafide", "bona_score"].to_numpy()
    spoof_scores = scored_df.loc[scored_df["label"] == "spoof", "bona_score"].to_numpy()
    eer, threshold = em.compute_eer(bona_scores, spoof_scores)

    return {
        "eer": float(eer * 100),
        "threshold": float(threshold),
        "accuracy": accuracy,
        "min_score": float(scored_df["bona_score"].min()),
        "max_score": float(scored_df["bona_score"].max()),
        "samples": int(len(scored_df)),
        "bonafide_samples": int((scored_df["label"] == "bonafide").sum()),
        "spoof_samples": int((scored_df["label"] == "spoof").sum()),
    }


def fuse_model_score_folders(
    folder_paths: Sequence[Union[str, Path]],
    reducer: Reducer = "mean",
    output_dir: Optional[Union[str, Path]] = None,
) -> Dict[str, Any]:
    """Fuse score files from multiple benchmark folders and compute metrics.

    Args:
        folder_paths: Iterable of directories produced by benchmark runs.
        reducer: Aggregation strategy. Either "mean" (default), "median", "max",
            "min", or a callable that accepts a 1D np.ndarray of scores.
        output_dir: Optional directory to store fused score files (per dataset).

    Returns:
        Dictionary containing fused DataFrames, dataset metrics, pooled metrics,
        and a summary DataFrame for quick inspection.
    """

    model_folders = [Path(p).resolve() for p in folder_paths]
    for folder in model_folders:
        if not folder.exists():
            raise FileNotFoundError(f"Folder does not exist: {folder}")

    catalogs = [_discover_datasets(folder) for folder in model_folders]
    if not catalogs:
        raise RuntimeError("No folders provided for fusion")

    dataset_sets = [set(cat.keys()) for cat in catalogs if cat]
    if not dataset_sets:
        raise RuntimeError("Unable to discover datasets for the provided folders")

    common_datasets = set.intersection(*dataset_sets)
    if not common_datasets:
        raise ValueError("No common datasets across the provided folders")

    protocol_map: Dict[str, Path] = {}
    dataset_score_paths: Dict[str, List[Path]] = {}

    for dataset in common_datasets:
        score_paths: List[Path] = []
        for catalog in catalogs:
            if dataset not in catalog:
                raise ValueError(f"Dataset '{dataset}' missing in one of the folders")
            meta = catalog[dataset]
            protocol_path = meta["protocol"]
            if dataset not in protocol_map:
                protocol_map[dataset] = protocol_path
            else:
                if protocol_map[dataset].resolve() != protocol_path.resolve():
                    raise ValueError(
                        f"Protocol mismatch for dataset '{dataset}':\n"
                        f"{protocol_map[dataset]} != {protocol_path}"
                    )
            score_paths.append(meta["score"])
        dataset_score_paths[dataset] = score_paths

    output_dir_path: Optional[Path] = None
    if output_dir is not None:
        output_dir_path = Path(output_dir).resolve()
        output_dir_path.mkdir(parents=True, exist_ok=True)

    fused_scores: Dict[str, pd.DataFrame] = {}
    fused_outputs: Dict[str, Optional[Path]] = {}
    dataset_metrics: Dict[str, Dict[str, Any]] = {}
    pooled_entries: List[pd.DataFrame] = []
    summary_rows: List[Dict[str, Any]] = []

    for dataset in sorted(common_datasets):
        merged_df: Optional[pd.DataFrame] = None
        for idx, score_path in enumerate(dataset_score_paths[dataset]):
            df = _read_score_file(score_path).rename(
                columns={
                    "spoof_score": f"spoof_{idx}",
                    "bona_score": f"bona_{idx}",
                }
            )
            merged_df = df if merged_df is None else merged_df.merge(df, on="path", how="inner")

        if merged_df is None:
            raise RuntimeError(f"No scores loaded for dataset '{dataset}'")

        spoof_cols = _select_columns("spoof_", merged_df.columns)
        bona_cols = _select_columns("bona_", merged_df.columns)

        fused_df = pd.DataFrame({"path": merged_df["path"]})
        fused_df["spoof_score"] = _apply_reducer(merged_df, spoof_cols, reducer)
        fused_df["bona_score"] = _apply_reducer(merged_df, bona_cols, reducer)
        fused_scores[dataset] = fused_df

        out_path = None
        if output_dir_path is not None:
            safe_name = dataset.replace(" ", "_")
            out_path = output_dir_path / f"{safe_name}_fusion.txt"
            fused_df.to_csv(
                out_path,
                sep=" ",
                header=False,
                index=False,
                float_format="%.6f",
            )
        fused_outputs[dataset] = out_path

        protocol_df = _read_protocol(protocol_map[dataset])
        scored_df = protocol_df.merge(fused_df, on="path", how="inner")
        missing = len(protocol_df) - len(scored_df)
        if missing > 0:
            print(
                f"⚠️ Dataset '{dataset}': {missing} entries dropped during fusion due to missing paths",
                flush=True,
            )

        metrics = _evaluate_scores(scored_df)
        metrics.update({
            "dataset": dataset,
            "missing_samples": int(missing),
        })

        dataset_metrics[dataset] = metrics
        summary_rows.append(metrics)
        pooled_entries.append(scored_df.assign(dataset=dataset))

    if not pooled_entries:
        raise RuntimeError("No scores available for pooled evaluation")

    pooled_df = pd.concat(pooled_entries, ignore_index=True)
    pooled_metrics = _evaluate_scores(pooled_df)
    pooled_metrics.update({
        "dataset": "POOLED",
        "datasets": len(common_datasets),
        "dataset_counts": {row["dataset"]: row["samples"] for row in summary_rows},
    })

    summary_df = pd.DataFrame(summary_rows).set_index("dataset").sort_index()

    return {
        "fused_scores": fused_scores,
        "fused_paths": fused_outputs,
        "dataset_metrics": dataset_metrics,
        "pooled_metrics": pooled_metrics,
        "summary": summary_df,
    }


In [ ]:
example_folders = [
    "/nvme2/hungdx/Lightning-hydra/logs/results/wildspoof_challenge_benchmark/XLSR_ConformerTCM_spoofceleb_clean_ft_new_noise_and_vocoded_Nov16", # 8.56
    "/nvme2/hungdx/Lightning-hydra/logs/results/wildspoof_challenge_benchmark/XLSR_ConformerTCM_spoofceleb_clean_ft_new_noise_balacne_ce_Nov16",  # 9.02
    "/nvme2/hungdx/Lightning-hydra/logs/results/wildspoof_challenge_benchmark/XLSR_ConformerTCM_spoofceleb_noise_ft", # 8.17
    "/nvme2/hungdx/Lightning-hydra/logs/results/wildspoof_challenge_benchmark/XLSR_ConformerTCM_spoofceleb_noise", # 8.39
    "/nvme2/hungdx/Lightning-hydra/logs/results/wildspoof_challenge_benchmark/xlsr_conformertcm_mdt_rawboost3_spoofceleb" # 10.53
]

# Uncomment to run the fusion pipeline end-to-end. This will load several large
# files, so expect it to take a few minutes.
fusion_result = fuse_model_score_folders(example_folders, reducer="mean")

# Show fusion EER and supporting metrics per dataset
summary_columns = ["eer", "threshold", "accuracy", "samples", "missing_samples"]
dataset_summary = (
    fusion_result["summary"][summary_columns]
    .sort_index()
    .assign(eer=lambda df: df["eer"].round(4), accuracy=lambda df: df["accuracy"].round(4))
)
display(dataset_summary)

# Pooled metrics across all datasets
fusion_result["pooled_metrics"]  # pooled EER/accuracy overview


,eer,threshold,accuracy,samples,missing_samples
dataset,,,,,
Wild_Spoof_Dataset,0.0527,-0.423771,99.9529,127367,0
asv19,20.7471,0.335293,73.0354,77713,0
df21,14.6479,-1.150263,91.2544,533928,0
in_the_wild,2.2594,-0.786056,97.6777,31779,0
shortcutASV,21.3056,-0.060269,79.2083,7200,0


{'eer': 7.561307162982739,
 'threshold': -0.09550390392541885,
 'accuracy': 91.00948987579484,
 'min_score': -5.038527011871338,
 'max_score': 4.370100021362305,
 'samples': 777987,
 'bonafide_samples': 91836,
 'spoof_samples': 686151,
 'dataset': 'POOLED',
 'datasets': 5,
 'dataset_counts': {'Wild_Spoof_Dataset': 127367,
  'asv19': 77713,
  'df21': 533928,
  'in_the_wild': 31779,
  'shortcutASV': 7200}}